# 05 — Barge-In: Real-time Interruption

## What is Barge-In?

**Barge-in** is the ability for a user to interrupt the model mid-response — just like interrupting someone in a real conversation.

Without barge-in, the assistant behaves like a voicemail: it plays its full response before listening again. With barge-in, the conversation feels **natural and bidirectional**.

### Why it matters

| Without barge-in | With barge-in |
|---|---|
| Model talks, user waits | User can speak at any time |
| Frustrating UX for long responses | Natural, human-like conversation |
| User must wait for error to correct it | User corrects immediately |
| One-way IVR feel | Two-way dialogue feel |

### How it works technically

Gemini Live handles interruption **server-side** using **Voice Activity Detection (VAD)**.
When the server detects the user is speaking while the model is responding:

1. The server **stops generating** the current response
2. It sets `server_content.interrupted = True` in the event stream
3. The session immediately starts listening to the user's new input

In this notebook we simulate interruptions programmatically using **activity signals**
(`ActivityStart` and `ActivityEnd`), which let you test barge-in without a microphone.

---
## What is VAD? (Voice Activity Detection)

VAD is an algorithm that detects whether audio contains speech or silence.

**Gemini Live uses server-side VAD**, which means:
- You stream raw PCM audio continuously
- The server decides when the user starts/stops speaking
- You don't need to manually segment speech

### How turns work with VAD:

```
Timeline:
  [User speaks] → VAD detects speech start
  [User speaking...]  → VAD buffers audio
  [User stops]  → VAD detects end-of-turn
  [Model responds] → server streams audio/text back
  [User speaks again] → VAD detects barge-in → model stops → listens
```

### Turn coverage modes:

The `turn_coverage` field in `RealtimeInputConfig` controls what audio is included in a turn:

| Mode | Behaviour |
|---|---|
| `TURN_INCLUDES_ALL_INPUT` | All audio sent since the last turn ends is included (default) |
| `TURN_INCLUDES_ONLY_ACTIVITY` | Only audio between `ActivityStart` and `ActivityEnd` signals is included |

Use `TURN_INCLUDES_ONLY_ACTIVITY` when you have your own VAD and want fine-grained control.

## Setup

In [ ]:
# !pip install -q google-genai numpy

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import os
import numpy as np
import IPython.display as ipd
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"

client = genai.Client(api_key=API_KEY)
print(f"SDK ready. Model: {MODEL}")

In [ ]:
def make_pcm(duration=2.0, rate=16000, freq=440):
    """Generate a sine-wave PCM clip (int16 bytes)."""
    t = np.linspace(0, duration, int(rate * duration))
    return (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16).tobytes()


def make_silence(duration=1.0, rate=16000):
    """Generate silent PCM bytes (zeros)."""
    samples = int(rate * duration)
    return np.zeros(samples, dtype=np.int16).tobytes()


def play_pcm(raw_bytes, rate=24000):
    """Return an IPython Audio widget from raw PCM bytes."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


print("Audio helpers ready.")

---
## Demo 1 — Simulating Barge-In with Activity Signals

We send a text query that prompts a long response,
then immediately send an `ActivityStart` signal (simulating the user speaking),
and watch for the `interrupted` flag in the model's response.

### How activity signals work:

```python
# Signal that user started speaking
await session.send_realtime_input(activity_start=types.ActivityStart())

# Signal that user stopped speaking
await session.send_realtime_input(activity_end=types.ActivityEnd())
```

These are manual overrides for the server-side VAD.

In [ ]:
async def demo_barge_in_signals():
    """
    Demo 1: Send activity signals to trigger barge-in detection.

    We start a long-form generation, then send activity_start to simulate
    the user beginning to speak. We detect 'interrupted' in the response.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        realtime_input_config=types.RealtimeInputConfig(
            turn_coverage="TURN_INCLUDES_ONLY_ACTIVITY",
        ),
    )

    print("=" * 60)
    print("Demo 1: Barge-In with Activity Signals")
    print("=" * 60)

    prompt = (
        "Please tell me a very detailed, long explanation of the history of Singapore "
        "from 1819 to the present day. Include as much detail as possible."
    )
    print(f"\nSending long-form query: '{prompt[:60]}...'")

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=prompt)

        async def send_barge_in():
            await asyncio.sleep(1.5)
            print("\n  [Sending ActivityStart — simulating user speaking]")
            await session.send_realtime_input(activity_start=types.ActivityStart())
            await asyncio.sleep(0.5)
            print("  [Sending ActivityEnd — simulating user stopped speaking]")
            await session.send_realtime_input(activity_end=types.ActivityEnd())
            await asyncio.sleep(0.2)
            print("  [Sending follow-up text after barge-in]")
            await session.send_realtime_input(text="Actually, just tell me the year Singapore was founded.")

        barge_in_task = asyncio.create_task(send_barge_in())

        chunks_received = 0
        interrupted_count = 0
        final_text = ""
        turn_count = 0

        async for resp in session.receive():
            if resp.server_content:
                sc = resp.server_content

                # Collect transcript chunks
                if sc.output_transcription and sc.output_transcription.text:
                    chunks_received += 1
                    if chunks_received <= 3:
                        print(f"  [Chunk {chunks_received}]: {sc.output_transcription.text[:60]}")
                    elif chunks_received == 4:
                        print("  [... more chunks ...]")

                # BARGE-IN DETECTED
                if sc.interrupted:
                    interrupted_count += 1
                    print(f"\n  BARGE-IN DETECTED (interrupted=True) — model stopped after {chunks_received} chunks")

                if sc.turn_complete:
                    turn_count += 1
                    if turn_count == 1 and interrupted_count == 0:
                        print("  [Turn 1 complete — no interruption occurred]")
                    elif turn_count >= 2:
                        print(f"\n  [Turn 2 response after barge-in]: {final_text}")
                        break

            if resp.go_away:
                print("Session ended by server.")
                break

        await barge_in_task

    print(f"\nSummary: {chunks_received} chunks received, {interrupted_count} interruption(s) detected")


asyncio.run(demo_barge_in_signals())


---
## Demo 2 — Detecting Interruptions in Audio Responses

This demo uses **audio output** so we can see how the `interrupted` flag cuts off audio generation.
We send synthetic PCM audio to simulate the user speaking, triggering the server-side VAD.

In [ ]:
async def demo_audio_interruption():
    """
    Demo 2: Audio-mode barge-in detection.

    We send a query that generates a long audio response,
    then stream synthetic speech audio to trigger the server VAD.
    We detect the interrupted flag and measure how much audio was received.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        realtime_input_config=types.RealtimeInputConfig(
            turn_coverage="TURN_INCLUDES_ONLY_ACTIVITY",
        ),
    )

    print("=" * 60)
    print("Demo 2: Audio Interruption Detection")
    print("=" * 60)

    long_prompt = (
        "Tell me a very long, detailed story about a dragon who learns to cook. "
        "Include many characters, plot twists, and vivid descriptions. Speak slowly and in full detail."
    )

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=long_prompt)
        print(f"\nSent prompt: '{long_prompt[:60]}...'")

        async def interrupt_after_delay():
            """Simulate user speaking (barge-in) 2 seconds into model response."""
            await asyncio.sleep(2.0)
            print("\n  [Simulating user barge-in with synthetic speech audio]")

            # Signal start of user speech
            await session.send_realtime_input(activity_start=types.ActivityStart())

            # Stream synthetic speech audio (sine wave at 440Hz)
            # In a real app, this would be mic audio
            speech_pcm = make_pcm(duration=1.0, rate=16000, freq=440)
            await session.send_realtime_input(
                audio=types.Blob(data=speech_pcm, mime_type="audio/pcm;rate=16000")
            )

            # Signal end of user speech
            await session.send_realtime_input(activity_end=types.ActivityEnd())
            print("  [User speech signal complete]")

            # Brief pause, then send the follow-up text
            await asyncio.sleep(0.3)
            await session.send_realtime_input(text="Wait, just tell me the dragon's name.")

        interrupt_task = asyncio.create_task(interrupt_after_delay())

        audio_chunks = []
        interrupted = False
        transcript_parts = []
        turn_count = 0

        async for resp in session.receive():
            # Collect audio chunks
            if resp.data:
                audio_chunks.append(resp.data)

            if resp.server_content:
                sc = resp.server_content

                # Collect transcription text
                if sc.output_transcription and sc.output_transcription.text:
                    transcript_parts.append(sc.output_transcription.text)

                # BARGE-IN DETECTED
                if sc.interrupted:
                    interrupted = True
                    total_audio = sum(len(c) for c in audio_chunks)
                    duration_s = total_audio / (2 * 24000)  # int16 samples at 24kHz
                    print(f"\n  BARGE-IN DETECTED — {total_audio:,} bytes ({duration_s:.2f}s) of audio received before interruption")

                if sc.turn_complete:
                    turn_count += 1
                    if turn_count >= 2:
                        print(f"\n  [Turn 2 complete — response after barge-in]")
                        break
                    elif not interrupted:
                        # Model finished without interruption
                        print("  [Turn 1 complete without interruption]")

            if resp.go_away:
                break

        await interrupt_task

    # Show the transcript of what was spoken before interruption
    partial_transcript = "".join(transcript_parts)
    print(f"\nPartial transcript before barge-in:")
    print(f"  '{partial_transcript[:200]}...'" if len(partial_transcript) > 200 else f"  '{partial_transcript}'")

    print(f"\nInterrupted: {interrupted}")

    if audio_chunks:
        raw = b"".join(audio_chunks)
        print(f"Total audio before barge-in: {len(raw):,} bytes")
        return play_pcm(raw, rate=24000)
    return None


audio_widget = asyncio.run(demo_audio_interruption())
audio_widget  # Play the partial audio to hear how far the model got

---
## Demo 3 — Turn Coverage Modes

The `turn_coverage` setting controls what audio counts as the user's "turn".
This has important implications for how barge-in works.

### Mode A: `TURN_INCLUDES_ALL_INPUT` (default)
Everything the server received since the last turn ended is included — even audio
that arrived while the model was speaking. The server uses its own VAD timing
to decide when a complete turn starts and ends.

**Best for**: Simple voice apps where you trust the server VAD.

### Mode B: `TURN_INCLUDES_ONLY_ACTIVITY`
Only audio explicitly bracketed by `ActivityStart` / `ActivityEnd` signals counts.
You have your own VAD running client-side and tell the server exactly which audio is speech.

**Best for**: Low-latency apps, noisy environments, or when you have a better VAD.

In [ ]:
async def demo_turn_coverage(mode: str, label: str):
    """
    Demo a specific turn_coverage mode.

    Args:
        mode: 'TURN_INCLUDES_ALL_INPUT' or 'TURN_INCLUDES_ONLY_ACTIVITY'
        label: Display name for the mode
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        realtime_input_config=types.RealtimeInputConfig(
            turn_coverage=mode,
        ),
    )

    print(f"\n{'=' * 60}")
    print(f"Mode: {label}")
    print(f"Config: turn_coverage='{mode}'")
    print("=" * 60)

    query = "Count from one to five slowly, saying each number on a new line."

    async with client.aio.live.connect(model=MODEL, config=config) as session:

        if mode == "TURN_INCLUDES_ONLY_ACTIVITY":
            await session.send_realtime_input(activity_start=types.ActivityStart())
            await session.send_realtime_input(text=query)
            await session.send_realtime_input(activity_end=types.ActivityEnd())
            print("Input bracketed with ActivityStart/ActivityEnd")
        else:
            await session.send_realtime_input(text=query)
            print("Input sent without activity signals")

        print(f"\nQuery: '{query}'")
        print("Response (transcript):")

        full_text = ""
        async for resp in session.receive():
            if resp.server_content:
                sc = resp.server_content
                if sc.output_transcription and sc.output_transcription.text:
                    full_text += sc.output_transcription.text
                    print(f"  {sc.output_transcription.text}", end="", flush=True)
                if sc.turn_complete:
                    break
            if resp.go_away:
                break

        print(f"\n\nFull transcript: {full_text}")


# Test Mode A: ALL_INPUT (default)
asyncio.run(demo_turn_coverage(
    mode="TURN_INCLUDES_ALL_INPUT",
    label="TURN_INCLUDES_ALL_INPUT (Default Mode)"
))


In [ ]:
# Test Mode B: ONLY_ACTIVITY (manual VAD control)
asyncio.run(demo_turn_coverage(
    mode="TURN_INCLUDES_ONLY_ACTIVITY",
    label="TURN_INCLUDES_ONLY_ACTIVITY (Manual VAD Mode)"
))

---
## Demo 4 — Full Barge-In Loop: Listen → Respond → Interrupt → New Response

This final demo shows the complete conversational barge-in loop:
1. User asks a question (text)
2. Model starts responding (we collect audio)
3. User interrupts mid-response (ActivityStart)
4. Model stops, we see `interrupted=True`
5. User sends their new question
6. Model responds to the new question

In [ ]:
async def demo_full_barge_in_loop():
    """
    Complete barge-in demonstration with state tracking.

    State machine:
      WAITING → model starts responding → RESPONDING
      RESPONDING → user interrupts → INTERRUPTED
      INTERRUPTED → user sends new question → RESPONDING_AGAIN
      RESPONDING_AGAIN → turn_complete → DONE
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        realtime_input_config=types.RealtimeInputConfig(
            turn_coverage="TURN_INCLUDES_ONLY_ACTIVITY",
        ),
    )

    print("=" * 60)
    print("Full Barge-In Loop Demo")
    print("=" * 60)

    initial_query = (
        "Explain the entire history of the Roman Empire in great detail, "
        "starting from Romulus and Remus through to the fall in 476 AD."
    )
    follow_up = "Actually, just when did Julius Caesar die?"

    state = "WAITING"
    text_so_far = ""
    turn_count = 0

    async with client.aio.live.connect(model=MODEL, config=config) as session:

        await session.send_realtime_input(activity_start=types.ActivityStart())
        await session.send_realtime_input(text=initial_query)
        await session.send_realtime_input(activity_end=types.ActivityEnd())
        print(f"[User]: {initial_query[:80]}...")
        state = "WAITING_FOR_RESPONSE"

        async def trigger_barge_in():
            """Wait for model to start, then barge in with a new question."""
            await asyncio.sleep(2.0)
            print("\n  --> [Barge-in triggered!]")
            await session.send_realtime_input(activity_start=types.ActivityStart())
            await asyncio.sleep(0.1)
            await session.send_realtime_input(activity_end=types.ActivityEnd())
            await asyncio.sleep(0.2)
            await session.send_realtime_input(activity_start=types.ActivityStart())
            await session.send_realtime_input(text=follow_up)
            await session.send_realtime_input(activity_end=types.ActivityEnd())
            print(f"  [User after barge-in]: {follow_up}")

        barge_task = asyncio.create_task(trigger_barge_in())

        async for resp in session.receive():
            if resp.server_content:
                sc = resp.server_content

                if sc.output_transcription and sc.output_transcription.text:
                    text_so_far += sc.output_transcription.text
                    state = "RESPONDING"

                if sc.interrupted:
                    state = "INTERRUPTED"
                    print(f"\n  [State: {state}] — '{text_so_far[:100]}...' was cut short")
                    text_so_far = ""

                if sc.turn_complete:
                    turn_count += 1
                    if state == "INTERRUPTED":
                        state = "WAITING_FOR_FOLLOW_UP"
                    elif turn_count >= 2:
                        state = "DONE"
                        print(f"\n[Model response to follow-up]: {text_so_far}")
                        break

            if resp.go_away:
                break

        await barge_task

    print(f"\nFinal state: {state}")
    print(f"Turns completed: {turn_count}")


asyncio.run(demo_full_barge_in_loop())


---
## API Reference — Barge-In Signals

```python
# --- Config ---
config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],  # or TEXT
    realtime_input_config=types.RealtimeInputConfig(
        turn_coverage="TURN_INCLUDES_ONLY_ACTIVITY",  # or TURN_INCLUDES_ALL_INPUT
    ),
    output_audio_transcription=types.AudioTranscriptionConfig(),
)

# --- Sending activity signals ---
# Signal user started speaking
await session.send_realtime_input(activity_start=types.ActivityStart())

# Stream audio (between start/end signals)
await session.send_realtime_input(
    audio=types.Blob(data=pcm_bytes, mime_type="audio/pcm;rate=16000")
)

# Signal user stopped speaking
await session.send_realtime_input(activity_end=types.ActivityEnd())

# --- Detecting barge-in in receive loop ---
async for resp in session.receive():
    if resp.data:  # Audio bytes from model
        audio_chunks.append(resp.data)

    if resp.server_content:
        sc = resp.server_content

        if sc.interrupted:  # Barge-in detected!
            print("Model was interrupted — stop playback, start recording")
            audio_chunks.clear()  # Discard unplayed audio

        if sc.turn_complete:  # Turn ended (normally or after interruption)
            break

    if resp.go_away:  # Server closing session
        break
```

### Key fields in receive events:

| Field | Type | Meaning |
|---|---|---|
| `resp.data` | `bytes` | Raw PCM audio from model |
| `resp.server_content.interrupted` | `bool` | True when model is interrupted |
| `resp.server_content.turn_complete` | `bool` | True when current turn is done |
| `resp.server_content.output_transcription.text` | `str` | Text transcript of audio |
| `resp.go_away` | `object` | Server is closing the session |

---
## Key Takeaways

1. **Barge-in is server-side by default** — Gemini Live's VAD detects user speech automatically during model output

2. **Detect interruption via `sc.interrupted`** — always check this flag in your receive loop so you can stop audio playback and clear buffers

3. **Activity signals for manual control** — use `ActivityStart` / `ActivityEnd` when you have your own VAD or want precise control

4. **Two turn_coverage modes**:
   - `TURN_INCLUDES_ALL_INPUT` — simple, trust the server VAD
   - `TURN_INCLUDES_ONLY_ACTIVITY` — precise, use your own VAD

5. **Discard buffered audio on interruption** — when `interrupted=True`, clear any audio you had queued for playback; the model has already stopped generating it

6. **The conversation continues seamlessly** — after an interruption, just send the user's new question normally; no need to reconnect

### Real-world implementation checklist:

- [ ] Check `sc.interrupted` in receive loop
- [ ] Stop audio playback immediately on interruption
- [ ] Clear unplayed audio buffer on interruption
- [ ] Start microphone capture after interruption
- [ ] Send `ActivityStart` when user begins speaking
- [ ] Stream mic audio as PCM chunks
- [ ] Send `ActivityEnd` when user stops speaking
- [ ] Resume receive loop for next model response